# Pertemuan 07 — Regresi Linear (Bagian 2)

Pada modul ini kita akan mendalami **geometri regresi linear** dan **regularisasi**.

> **Kelanjutan Pertemuan 06.** Sebelumnya kita sudah mengenal bentuk model regresi linear, fungsi basis, dan tafsiran geometrisnya. Pertemuan ini memperdalam sisi geometrinya, lalu masuk ke persoalan yang muncul di praktik: **overfitting** dan **ill-conditioning**, beserta obat untuk keduanya, yaitu regularisasi.

**Prasyarat:** memahami regresi linear dan fungsi basis (Pertemuan 06), serta konsep overfitting (Pertemuan 03).


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from plotly.subplots import make_subplots
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

import warnings
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)

np.random.seed(42)

## Tujuan Pembelajaran

Notebook ini disusun untuk memberikan pemahaman menyeluruh mengenai **regresi linear** beserta perluasannya, dengan penekanan pada konsep-konsep berikut.

**1. Geometri regresi linear**
- Menelusuri **tafsiran geometris** regresi linear, mencakup:
    - **Rentang fitur masukan** (*span*): memahami bahwa prediksi selalu terletak di dalam subruang yang dibentangkan oleh fitur masukan.
    - **Residual**: selisih antara nilai prediksi dan nilai target sebenarnya.
    - **Proyeksi ortogonal**: bagaimana garis terbaik meminimalkan residual dengan memproyeksikan vektor target ke ruang fitur.

**2. Analisis residual**
- Mempelajari cara menganalisis **residual** untuk menilai mutu model regresi.
- Memahami pola pada plot residual guna mengenali masalah seperti **heteroskedastisitas** atau hubungan **non-linear**.

**3. Regularisasi**
- Mengenal **regularisasi L2 (Ridge Regression)** dan **regularisasi L1 (Lasso Regression)** untuk mengatasi overfitting dan ill-conditioning.
- Memvisualkan pengaruh regularisasi terhadap fungsi kerugian, koefisien model, dan prediksinya.
- Membandingkan perilaku Ridge dan Lasso pada berbagai kekuatan regularisasi.

**4. Dampak regularisasi**
- Menunjukkan pengaruh regularisasi terhadap kinerja model, khususnya dalam hal:
    - **Penyusutan koefisien** (*coefficient shrinkage*)
    - **Kompleksitas model**

**5. Ill-conditioning**
- Memahami masalah yang timbul bila fitur pada matriks rancangan hampir **kolinear**.
- Menunjukkan bagaimana regularisasi (Ridge) dapat meredam dampak ill-conditioning dan memperbaiki kestabilan numeriknya.

> **Cara memakai modul ini.** Jalankan sel kode berurutan dari atas ke bawah. Beberapa grafik bersifat interaktif dengan penggeser $\lambda$ — geser dan amati perubahannya, karena di situlah inti pemahamannya.


---
## 1. Geometri Regresi Linear

Bagian ini menyajikan regresi linear dari sudut pandang **aljabar linear**. Sudut pandang ini menjelaskan *mengapa* rumus penyelesaian regresi linear berbentuk seperti yang kita kenal.

Pada sel berikutnya kita akan menampilkan:

* **Residual**: selisih antara nilai prediksi dan nilai target yang sebenarnya.
* **Rentang fitur masukan** ($\text{span}(\mathbb{X})$): subruang yang dibentuk oleh semua kombinasi linear dari vektor-vektor kolom matriks fitur.

**Prediksi sebagai kombinasi linear**

Prediksi $\mathbf{Y}(\mathbb{X}, \mathbf{w}) = \mathbb{X}\mathbf{w}$ adalah **kombinasi linear dari kolom-kolom** $\mathbb{X}$. Artinya vektor prediksi $\mathbf{Y}$ selalu terletak di dalam **rentang** matriks masukan, ditulis $\text{span}(\mathbb{X})$.

**Tafsirannya**

* Vektor prediksi $\mathbf{Y}$ **selalu** berada di dalam $\text{span}(\mathbb{X})$, sekalipun vektor target sebenarnya $\mathbf{t}$ berada di luarnya.
* Tujuan regresi linear adalah menemukan vektor $\mathbf{Y}$ di dalam $\text{span}(\mathbb{X})$ yang **paling dekat** dengan $\mathbf{t}$.

**Meminimalkan residual**

* Vektor residual $\mathbf{e} = \mathbf{t} - \mathbf{Y}$ menyatakan selisih antara target sebenarnya dan prediksi.
* Meminimalkan jarak antara $\mathbf{Y}$ dan $\mathbf{t}$ berarti meminimalkan **panjang** vektor residual $\mathbf{e}$.

**Proyeksi ortogonal**

* Panjang $\mathbf{e}$ menjadi minimum tepat ketika $\mathbf{Y}$ merupakan **proyeksi ortogonal** dari $\mathbf{t}$ ke $\text{span}(\mathbb{X})$.
* Dengan demikian $\mathbf{e}$ tegak lurus terhadap $\text{span}(\mathbb{X})$ — inilah syarat kecocokan terbaik pada regresi linear.

> **Analogi sederhana.** Bayangkan $\text{span}(\mathbb{X})$ sebagai lantai sebuah ruangan, dan $\mathbf{t}$ sebagai titik yang melayang di udara. Titik di lantai yang paling dekat dengan titik melayang itu adalah **bayangannya tepat di bawahnya**, dan garis penghubungnya tegak lurus terhadap lantai. Bayangan itulah $\mathbf{Y}$, dan garis tegak lurusnya adalah residual $\mathbf{e}$.

Putar grafik tiga dimensi berikut dengan tetikus untuk memastikan bahwa panah oranye benar-benar tegak lurus terhadap bidangnya.


In [ ]:
# ============ Parameter tampilan ============
SCALE        = 2.0
LINE_W       = 14
HEAD_SIZE_T  = 1.9
HEAD_SIZE_E  = 1.5
COEFF_RANGE  = 3.0
GRID_LINES   = 11
PAD_RATIO    = 0.20
BRACKET_FRAC = 0.22
# =================================================

x1 = np.array([3.0, 0.4, 0.6]) * SCALE
x2 = np.array([1.2, 2.4, -0.5]) * SCALE

w1, w2 = 1.5, 0.9
residual_height = 2.2 * SCALE

O = np.zeros(3)
Y = w1*x1 + w2*x2

def unit(v):
    n = np.linalg.norm(v);  return v/n if n else v

def gram_schmidt(a, b):
    u1 = unit(a)
    b_perp = b - np.dot(b, u1)*u1
    u2 = unit(b_perp)
    return u1, u2

u_hat, v_hat = gram_schmidt(x1, x2)
n_hat = unit(np.cross(u_hat, v_hat))

t = Y + residual_height*n_hat

nu = nv = 45
U = np.linspace(-COEFF_RANGE, COEFF_RANGE, nu)
V = np.linspace(-COEFF_RANGE, COEFF_RANGE, nv)
UU, VV = np.meshgrid(U, V)
P = UU[...,None]*x1 + VV[...,None]*x2

plane = go.Surface(
    x=P[...,0], y=P[...,1], z=P[...,2],
    opacity=0.22, showscale=False, name="span(X)",
    surfacecolor=np.zeros_like(P[...,0]),
    colorscale=[[0,"green"],[1,"green"]],
    hoverinfo="skip"
)

grid = []
for s in np.linspace(-COEFF_RANGE, COEFF_RANGE, GRID_LINES):
    a, b = s*x1 + (-COEFF_RANGE)*x2, s*x1 + (COEFF_RANGE)*x2
    grid.append(go.Scatter3d(x=[a[0],b[0]], y=[a[1],b[1]], z=[a[2],b[2]],
                             mode="lines", line=dict(width=2, color="rgba(0,90,0,0.6)"),
                             showlegend=False, hoverinfo="skip"))
    a, b = (-COEFF_RANGE)*x1 + s*x2, (COEFF_RANGE)*x1 + s*x2
    grid.append(go.Scatter3d(x=[a[0],b[0]], y=[a[1],b[1]], z=[a[2],b[2]],
                             mode="lines", line=dict(width=2, color="rgba(0,90,0,0.6)"),
                             showlegend=False, hoverinfo="skip"))


line_traces, cone_traces = [], []

def arrow(start, end, name, color, width=LINE_W, head=1.6, back=None, dash=None, show_legend=True):
    vec = end - start
    L = np.linalg.norm(vec)
    if L == 0: return
    if back is None:
        back = max(0.9*head, 0.02*L)
    tip_base = end - back*(vec/L)

    line_traces.append(go.Scatter3d(
        x=[start[0], tip_base[0]], y=[start[1], tip_base[1]], z=[start[2], tip_base[2]],
        mode="lines",
        line=dict(width=width, color=color, dash=dash) if dash else dict(width=width, color=color),
        name=name, showlegend=show_legend, hoverinfo="skip"
    ))
    cone_traces.append(go.Cone(
        x=[end[0]], y=[end[1]], z=[end[2]],
        u=[vec[0]], v=[vec[1]], w=[vec[2]],
        sizemode="absolute", sizeref=head, anchor="tip",
        showscale=False, colorscale=[[0, color],[1, color]],
        name=name, showlegend=False
    ))

# vektor-vektor kolom (basis dari span)
arrow(O, x1, "X·,1", "darkgreen")
arrow(O, x2, "X·,2", "darkgreen")

# Y (vektor prediksi)
arrow(O, Y, "Y = Xw", "black")

# t
arrow(O, t, "t", "#d62728", head=HEAD_SIZE_T)

# vektor residual e
arrow(Y, t, "e = t − Xw", "#ff9800", head=HEAD_SIZE_E, width=LINE_W-2)

# Penanda sudut siku-siku (bukti ketegaklurusan)
y_dir = unit(Y - O)
e_dir = unit(t - Y)

tick  = BRACKET_FRAC * min(np.linalg.norm(x1), np.linalg.norm(x2)) * SCALE
h = tick

base   = Y - tick * y_dir
corner = base + h * e_dir
end_on_residual = Y + h * e_dir

bracket = go.Scatter3d(
    x=[base[0], corner[0], end_on_residual[0]],
    y=[base[1], corner[1], end_on_residual[1]],
    z=[base[2], corner[2], end_on_residual[2]],
    mode="lines",
    line=dict(width=10, color="royalblue"),
    showlegend=False, hoverinfo="skip"
)

p1 = Y - tick * y_dir
p2 = p1 + tick * e_dir

pts = np.vstack([O, x1, x2, Y, t, p1, p2])
mins, maxs = pts.min(axis=0), pts.max(axis=0)
pad = PAD_RATIO * np.max(maxs - mins if np.any(maxs - mins) else np.array([1,1,1]))
xr, yr, zr = [mins[0]-pad, maxs[0]+pad], [mins[1]-pad, maxs[1]+pad], [mins[2]-pad, maxs[2]+pad]

# label pada gambar
def label(pt, text, d=0.12*SCALE):
    return dict(x=pt[0]+d, y=pt[1]+d, z=pt[2]+d, text=text,
                showarrow=False, bgcolor="rgba(255,255,255,0.85)", bordercolor="black")
ann = [label(x1,"X<sub>·,1</sub>"), label(x2,"X<sub>·,2</sub>"),
       label(Y,"Y = Xw"), label(t,"t"), label((Y+t)/2,"e")]

fig = go.Figure(data=[plane, *grid, bracket, *line_traces, *cone_traces])  # kerucut digambar terakhir

fig.update_layout(
    title="Geometri Regresi Linear: Proyeksi Ortogonal",
    width=1100, height=900,
    scene=dict(
        xaxis=dict(title="x", range=xr, zeroline=False, showbackground=False),
        yaxis=dict(title="y", range=xr, zeroline=False, showbackground=False),
        zaxis=dict(title="z", range=xr, zeroline=False, showbackground=False),
        annotations=ann,
        aspectmode="cube",
        camera=dict(eye=dict(x=1, y=-1, z=1.4))
    ),
    legend=dict(x=0.02, y=0.98, bgcolor="rgba(255,255,255,0.75)")
)

fig.show()

Bacalah gambar di atas seperti berikut:

* Dua panah **hijau** adalah vektor kolom matriks $\mathbb{X}$. Keduanya membentangkan sebuah bidang, yaitu $\text{span}(\mathbb{X})$.
* Panah **hitam** adalah prediksi $\mathbf{Y} = \mathbb{X}\mathbf{w}$, yang selalu terletak **di dalam** bidang itu.
* Panah **merah** adalah target sebenarnya $\mathbf{t}$, yang umumnya berada **di luar** bidang.
* Panah **oranye** adalah residual $\mathbf{e} = \mathbf{t} - \mathbf{Y}$, dan penanda siku-siku biru menunjukkan bahwa panah ini **tegak lurus** terhadap bidangnya.

> **Inilah inti regresi kuadrat terkecil.** Karena $\mathbf{t}$ pada umumnya tidak dapat dicapai secara persis, kita mengambil titik terdekat yang bisa dicapai, yaitu proyeksi ortogonalnya. Syarat ketegaklurusan $\mathbb{X}^T\mathbf{e} = 0$ inilah yang bila diuraikan menghasilkan persamaan normal $\mathbf{w} = (\mathbb{X}^T\mathbb{X})^{-1}\mathbb{X}^T\mathbf{t}$.

---
## 2. Analisis Residual

**Residual** adalah selisih antara nilai yang teramati (target sebenarnya) dan nilai prediksi model. Menganalisis residual membantu kita menilai kecocokan model sekaligus mengenali masalah seperti hubungan non-linear, heteroskedastisitas, atau pencilan.

Pada sel berikutnya kita akan melihat dua hal.

**1. Plot residual**

* Grafik pencar antara residual (sumbu-y) dan nilai prediksi (sumbu-x).
* Idealnya residual **tersebar acak di sekitar nol** tanpa pola apa pun, yang menandakan model sudah menangkap seluruh struktur data.
* Bila terlihat pola melengkung, berarti masih ada hubungan non-linear yang belum tertangkap.

**2. Heteroskedastisitas**

* Terjadi ketika **ragam residual tidak tetap** di sepanjang rentang nilai prediksi.
* Sering tampak sebagai pola **menyerupai kipas** yang melebar pada plot residual.
* Menandakan bahwa ketidakpastian prediksi berbeda-beda pada tiap rentang nilai.

> **Mengapa penting?** Banyak uji statistik pada regresi mengandaikan ragam residual yang tetap (*homoskedastisitas*). Bila andaian itu dilanggar, prediksinya masih boleh dipakai, tetapi ukuran ketidakpastiannya menjadi tidak dapat dipercaya.

Kita memakai dataset **California Housing** untuk melatih model regresi linear sederhana: memprediksi **nilai tengah harga rumah** dari **pendapatan tengah** penduduk suatu blok. Model dievaluasi dengan MSE dan R-kuadrat pada data uji.


In [ ]:
data = fetch_california_housing(as_frame=True)
df = data.frame.copy()

df = df[["MedInc", "MedHouseVal"]].rename(columns={"MedHouseVal": "y"})

# Menyaring data untuk membuang pencilan ekstrem
df = df[(df["MedInc"] < 10) & (df["y"] < 5)]

# Sampel berstrata menurut kuantil MedInc: cakupan tetap luas, tampilan tidak terlalu padat
np.random.seed(7)
q = pd.qcut(df["MedInc"], q=15, duplicates="drop")
sampled = (
    df.groupby(q, observed=True)
      .apply(lambda g: g.sample(n=min(len(g), 30), random_state=7))
      .reset_index(drop=True)
)

X = sampled[["MedInc"]].values
y = sampled["y"].values

# Membagi data latih dan data uji
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Melatih model regresi linear
model = LinearRegression()
model.fit(X_train, y_train)

y_pred_train = model.predict(X_train)
y_pred_test  = model.predict(X_test)

resid_train = y_train - y_pred_train
resid_test  = y_test  - y_pred_test

# Menghitung metrik pada data uji
mse = mean_squared_error(y_test, y_pred_test)
r2  = r2_score(y_test, y_pred_test)
n, p = X_test.shape[0], X_test.shape[1]
adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)

print(f"MSE data uji        : {mse:.4f}")
print(f"R-kuadrat data uji  : {r2:.4f}")
print(f"R-kuadrat terkoreksi: {adj_r2:.4f}")

def binned_trend(x, y, bins=12):
    edges = np.linspace(np.min(x), np.max(x), bins + 1)
    idx = np.digitize(x, edges) - 1
    xc = []
    yc = []
    for b in range(bins):
        mask = idx == b
        if np.any(mask):
            xc.append(np.mean(x[mask]))
            yc.append(np.mean(y[mask]))
    return np.array(xc), np.array(yc)

# Grafik 1: data dan garis hasil pelatihan
fig, axs = plt.subplots(1, 2, figsize=(12, 5))
axs[0].scatter(X_train, y_train, s=50, alpha=0.7, edgecolor="k", color="crimson")
xline = np.linspace(X_train.min(), X_train.max(), 200).reshape(-1, 1)
yline = model.predict(xline)
axs[0].plot(xline, yline, lw=3, label="Garis hasil pelatihan", color="blue", zorder=3)
axs[0].set_xlabel("Pendapatan tengah (MedInc)", fontsize=14)
axs[0].set_ylabel("Nilai tengah harga rumah (y)", fontsize=14)
axs[0].set_title("Regresi Linear Sederhana: y ~ MedInc", fontsize=16, fontweight="bold")
axs[0].legend(frameon=False, fontsize=12)
axs[0].grid(True, linestyle="--", alpha=0.6)

# Grafik 2: residual terhadap nilai prediksi
axs[1].scatter(y_pred_train, resid_train, s=50, alpha=0.7, edgecolor="k", color="crimson")
axs[1].axhline(0, color="k", lw=2, ls="--", label="Garis residual nol")
axs[1].set_xlabel("Nilai prediksi", fontsize=14)
axs[1].set_ylabel("Residual", fontsize=14)
axs[1].set_title("Residual vs Nilai Prediksi\nPola kipas → heteroskedastisitas", fontsize=16, fontweight="bold")
axs[1].legend(frameon=False, fontsize=12)
axs[1].grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()

**Bacalah kedua grafik.**

Grafik kiri menunjukkan garis regresi menembus awan titik data. Kecenderungan naik terlihat jelas: semakin tinggi pendapatan, semakin tinggi harga rumah.

Grafik kanan adalah **plot residual**. Perhatikan bentuknya yang melebar seperti kipas ke arah kanan — inilah **heteroskedastisitas**. Artinya prediksi model jauh lebih tidak pasti pada daerah berpendapatan tinggi dibanding daerah berpendapatan rendah.

Perhatikan pula nilai R² yang tercetak: itulah bagian ragam harga rumah yang berhasil dijelaskan oleh pendapatan saja. Sisanya berasal dari faktor lain yang belum masuk model — lokasi, usia bangunan, jumlah kamar, dan sebagainya.

---
## 3. Regularisasi

**Regularisasi** dirancang untuk mencegah overfitting dengan menambahkan **suku penalti** pada fungsi kerugian. Penalti ini membuat model enggan memberikan bobot yang terlalu besar kepada fitur, sehingga kemampuan generalisasinya membaik.

Fungsi tujuan yang diminimalkan menjadi:

$$
E(\mathbf{w}) = \underbrace{E_D(\mathbf{w})}_{\text{galat pada data}} + \lambda \underbrace{E_W(\mathbf{w})}_{\text{penalti bobot}}
$$

dengan $\lambda \ge 0$ mengatur seberapa kuat penaltinya.

| Nilai $\lambda$ | Akibatnya |
|---|---|
| $\lambda = 0$ | Tidak ada penalti; kembali menjadi regresi linear biasa |
| $\lambda$ kecil | Penalti ringan; model masih bebas mengikuti data |
| $\lambda$ besar | Penalti berat; seluruh bobot tertekan mendekati nol |
| $\lambda \to \infty$ | Semua bobot menjadi nol; model hanya menebak nilai rata-rata |

Dua jenis regularisasi yang paling umum adalah **L2 (Ridge Regression)** dan **L1 (Lasso Regression)**.

**Ridge Regression** meminimalkan jumlah kuadrat residual dengan tambahan penalti yang sebanding dengan **kuadrat** besar koefisiennya:

$$
E_W(\mathbf{w}) = \tfrac{1}{2}\|\mathbf{w}\|_2^2 = \tfrac{1}{2}\sum_j w_j^2
$$

Akibatnya seluruh koefisien **menyusut** bersama-sama, tetapi jarang benar-benar mencapai nol.

**Lasso Regression** memakai **nilai mutlak** koefisien sebagai penaltinya:

$$
E_W(\mathbf{w}) = \|\mathbf{w}\|_1 = \sum_j |w_j|
$$

Bentuk ini dapat menyusutkan sebagian koefisien hingga **tepat nol**, sehingga sekaligus berperan sebagai **pemilihan fitur** (*feature selection*).

| | Ridge (L2) | Lasso (L1) |
|---|---|---|
| Bentuk penalti | $\sum w_j^2$ | $\sum \lvert w_j \rvert$ |
| Bentuk daerah batasnya | Lingkaran | Belah ketupat (bersudut) |
| Efek pada koefisien | Menyusut merata | Sebagian menjadi tepat nol |
| Pemilihan fitur | Tidak | Ya |
| Paling cocok saat | Semua fitur sedikit banyak berguna | Hanya sebagian fitur yang penting |

> **Mengapa Lasso bisa menghasilkan nol persis?** Karena daerah batas L1 berbentuk **belah ketupat yang bersudut tajam**, dan sudut-sudutnya tepat berada pada sumbu koordinat. Kontur galat cenderung menyentuh daerah itu di sudutnya, dan titik sudut berarti salah satu koefisiennya nol. Bandingkan dengan daerah batas L2 yang berbentuk **lingkaran mulus** tanpa sudut, sehingga titik singgungnya hampir tidak pernah tepat di sumbu. Perbedaan bentuk inilah yang akan Anda lihat pada dua visualisasi berikutnya.


### 3.1 Regularisasi L2: Ridge Regression

Grafik berikut menampilkan kontur $E_D(\mathbf{w})$, yaitu galat pada data, dalam ruang parameter dua dimensi ($w_1$ dan $w_2$).

* **Bintang** menandai solusi tanpa regularisasi, yaitu titik terendah kontur galat.
* **Lingkaran ungu** adalah daerah batas L2, yakni himpunan titik dengan $\|\mathbf{w}\|_2 = c(\lambda)$.
* **Silang biru kehijauan** adalah solusi Ridge $\hat{\mathbf{w}}(\lambda)$.

Semakin besar $\lambda$, semakin kecil jari-jari lingkarannya, dan solusinya semakin tertarik mendekati titik asal.


In [ ]:
def plot_E_D():
    w0 = np.array([3.2, 0.9])
    theta = np.deg2rad(25)
    R = np.array([[np.cos(theta), -np.sin(theta)],
                    [np.sin(theta),  np.cos(theta)]])
    A = R @ np.diag([6.0, 1.0]) @ R.T * 2.5
    # ---------- Kisi untuk panel kiri ----------
    x = np.linspace(-10.0, 10.0, 501)
    y = np.linspace(-10.0,  10.0, 501)
    X, Y = np.meshgrid(x, y)
    U = np.stack([X - w0[0], Y - w0[1]], axis=-1)
    Z_ED = 0.5 * np.einsum("...i,...i", U, U @ A)

    zmin, zmax = 0.0, float(np.percentile(Z_ED, 95))

    fig = go.Figure()

    # Panel kiri: gradasi biru beserta garis konturnya
    level_start = 0.01 * zmax
    level_end   = 0.99 * zmax
    num_levels  = 20
    level_step  = (level_end - level_start) / num_levels

    fig.add_trace(go.Contour(
        x=x, y=y, z=np.clip(Z_ED, zmin, zmax),
        zmin=zmin, zmax=zmax,
        colorscale="Blues", reversescale=True,
        contours=dict(start=level_start, end=level_end, size=level_step,
                    showlines=False),
        connectgaps=True,
        showscale=False, opacity=0.96
    ))

    # Solusi tanpa regularisasi (tanda bintang)
    fig.add_trace(
        go.Scatter(
            x=[w0[0]], y=[w0[1]],
            mode="markers+text",
            marker=dict(symbol="star", size=14, color="crimson", line=dict(width=1)),
            textposition="top right",
            name="ŵ(λ=0) — tanpa regularisasi"
        )
    )

    fig.update_layout(
        width=600, height=650,
        template="plotly_white",
        title="Kontur Galat Data E_D beserta Solusi Tanpa Regularisasi",
        xaxis_title="w₁",
        yaxis_title="w₂",
        xaxis=dict(scaleanchor="y", scaleratio=1,
                    zeroline=False, range=[x.min(), x.max()], title=dict(standoff=20)),
        yaxis=dict(zeroline=False, range=[y.min(), y.max()], title=dict(standoff=20)),
        showlegend=False
    )
    fig.update_xaxes(range=[w0[0]-10, w0[0]+10])
    fig.update_yaxes(range=[w0[1]-10, w0[1]+10])

    return fig

plot_E_D().show()

In [ ]:
# Solusi Ridge memiliki rumus tertutup: w = (A + λI)^{-1} A w0
def ridge_solution(lmbda, A, w0):
    w_hat = np.linalg.solve(A + lmbda*np.eye(2), A @ w0)
    ED = 0.5 * (w_hat - w0) @ A @ (w_hat - w0)
    EW = 0.5 * (w_hat @ w_hat)
    return w_hat, ED, EW, ED + lmbda*EW

In [ ]:
def lasso_objective(w, A, w0, lmbda):
    # E_D(w) = 0.5 * (w - w0)ᵀ A (w - w0)  → galat pada data
    ED = 0.5 * (w - w0).T @ A @ (w - w0)
    # E_W(w) = ||w||_1 = |w1| + |w2|  → penalti L1
    EW_l1 = np.sum(np.abs(w))
    # Fungsi tujuan = E_D(w) + lambda * E_W(w)
    return ED + lmbda * EW_l1

In [ ]:
# Lasso tidak punya rumus tertutup, sehingga diselesaikan secara numerik
from scipy.optimize import minimize
def lasso_solution(lmbda, A, w0):
    initial_guess = np.zeros(2)

    result = minimize(lasso_objective, initial_guess, args=(A, w0, lmbda), method='Nelder-Mead', tol=1e-6)

    w_hat = result.x
    ED = 0.5 * (w_hat - w0).T @ A @ (w_hat - w0)
    EW_l1 = np.sum(np.abs(w_hat))
    J = ED + lmbda * EW_l1

    return w_hat, ED, EW_l1, J

In [ ]:
w0 = np.array([3.2, 0.9])
theta = np.deg2rad(25)
R = np.array([[np.cos(theta), -np.sin(theta)],
                [np.sin(theta),  np.cos(theta)]])
A = R @ np.diag([6.0, 1.0]) @ R.T * 2.5

fig = plot_E_D()

lam = 0.0

# ---------- Unsur tetap: daerah batas dan titik solusi pada lambda terpilih ----------
w, ED, EW, J = ridge_solution(lam, A, w0)
c = np.linalg.norm(w)
t = np.linspace(0, 2*np.pi, 400)
cx, cy = c*np.cos(t), c*np.sin(t)

fig.add_trace(go.Scatter(x=cx, y=cy, mode="lines",
                       line=dict(width=5, color="darkmagenta"), name="‖w‖₂ = c(λ)"))
fig.add_trace(go.Scatter(x=[w[0]], y=[w[1]], mode="markers",
                       marker=dict(size=12, symbol="x", color="teal"), name="ŵ(λ)"))

fig.show()

Pada $\lambda = 0$ lingkarannya melalui titik solusi tanpa regularisasi, sehingga hasilnya sama persis dengan regresi linear biasa.

Sel berikutnya menjadikan $\lambda$ dapat digeser sehingga perubahannya dapat diamati langsung.

In [ ]:
def plot_reg_with_regularization(func):
    w0 = np.array([3.2, 0.9])
    theta = np.deg2rad(25)
    R = np.array([[np.cos(theta), -np.sin(theta)],
                [np.sin(theta),  np.cos(theta)]])
    A = R @ np.diag([6.0, 1.0]) @ R.T * 2.5
    x = np.linspace(-20.0, 20.0, 501)
    y = np.linspace(-10.0,  10.0, 501)
    X, Y = np.meshgrid(x, y)
    U = np.stack([X - w0[0], Y - w0[1]], axis=-1)
    Z_ED = 0.5 * np.einsum("...i,...i", U, U @ A)
    zmin, zmax = 0.0, float(np.percentile(Z_ED, 95))
    lams       = np.linspace(0.0, 15.0, 16)
    lams_curve = np.linspace(0.0, 15.0, 600)
    ED_curve, EW_curve = [], []
    for lam in lams_curve:
        _, ED, EW, _ = func(lam, A, w0)
        ED_curve.append(ED); EW_curve.append(EW)
    ED_curve = np.array(ED_curve); EW_curve = np.array(EW_curve)
    E_curve  = ED_curve + lams_curve * EW_curve
    ymax = float(E_curve.max()) * 1.05
    fig = make_subplots(
    rows=1, cols=2, column_widths=[0.55, 0.45],
    specs=[[{"type":"contour"}, {"type":"xy"}]]
    )
    fig.update_layout(template="plotly_white")

    level_start = 0.01 * zmax
    level_end   = 0.99 * zmax
    num_levels  = 20
    level_step  = (level_end - level_start) / num_levels

    fig.add_trace(go.Contour(
        x=x, y=y, z=np.clip(Z_ED, zmin, zmax),
        zmin=zmin, zmax=zmax,
        colorscale="Blues", reversescale=True,
        contours=dict(start=level_start, end=level_end, size=level_step,
                    showlines=False),
        connectgaps=True,
        showscale=False, opacity=0.96
    ), row=1, col=1)


    fig.add_trace(
        go.Scatter(
            x=[w0[0]], y=[w0[1]],
            mode="markers+text",
            marker=dict(symbol="star", size=14, color="crimson", line=dict(width=1)),
            textposition="top right",
            name="ŵ(λ=0) — tanpa regularisasi"
        ),
        row=1, col=1
    )

    # Panel kanan: kurva E_D, λE_W, dan E terhadap λ
    fig.add_trace(go.Scatter(x=lams_curve, y=ED_curve,            mode="lines", line=dict(width=3),
                            name="E_D( ŵ(λ) )"), row=1, col=2)
    fig.add_trace(go.Scatter(x=lams_curve, y=lams_curve*EW_curve, mode="lines", line=dict(width=3),
                            name="λ · E_W( ŵ(λ) )"), row=1, col=2)
    fig.add_trace(go.Scatter(x=lams_curve, y=E_curve,             mode="lines", line=dict(width=3),
                            name="E( ŵ(λ) )"), row=1, col=2)

    # ---------- Unsur yang berubah mengikuti λ ----------

    w, ED, EW, J = func(lam, A, w0)
    if func is ridge_solution:
        c = np.linalg.norm(w)
        t = np.linspace(0, 2*np.pi, 400)
        cx, cy = c*np.cos(t), c*np.sin(t)
    elif func is lasso_solution:
        c = np.linalg.norm(w, ord=1)
        cx = c * np.array([1, 0, -1, 0, 1])
        cy = c * np.array([0, 1, 0, -1, 0])

    shape_tr = go.Scatter(x=cx, y=cy, mode="lines",
                        line=dict(width=5, color="darkmagenta"), name="‖w‖₂ = c(λ)")
    opt_tr    = go.Scatter(x=[], y=[], mode="markers",
                        marker=dict(size=12, symbol="x", color="teal"), name="ŵ(λ)")
    vline_tr  = go.Scatter(x=[], y=[], mode="lines",
                        line=dict(width=2, dash="dot", color="teal"), showlegend=False)
    ed_mk_tr  = go.Scatter(x=[], y=[], mode="markers+text", marker=dict(size=9),
                        text=[""], textposition="top center", showlegend=False)
    lew_mk_tr = go.Scatter(x=[], y=[], mode="markers+text", marker=dict(size=9),
                        text=[""], textposition="top center", showlegend=False)
    e_mk_tr   = go.Scatter(x=[], y=[], mode="markers+text", marker=dict(size=9),
                        text=[""], textposition="top center", showlegend=False)

    fig.add_trace(shape_tr, row=1, col=1)
    fig.add_trace(opt_tr,    row=1, col=1)
    fig.add_trace(vline_tr,  row=1, col=2)
    fig.add_trace(ed_mk_tr,  row=1, col=2)
    fig.add_trace(lew_mk_tr, row=1, col=2)
    fig.add_trace(e_mk_tr,   row=1, col=2)

    dyn_ix = list(range(len(fig.data)-6, len(fig.data)))

    # ---------- Bingkai animasi untuk tiap nilai λ ----------
    frames = []
    for lam in lams:
        w, ED, EW, E = func(lam, A, w0)
        if func is ridge_solution:
            c = np.linalg.norm(w)
            t = np.linspace(0, 2*np.pi, 400)
            cx, cy = c*np.cos(t), c*np.sin(t)
        elif func is lasso_solution:
            c = np.linalg.norm(w, ord=1)
            cx = c * np.array([1, 0, -1, 0, 1])
            cy = c * np.array([0, 1, 0, -1, 0])
        idx = int(np.argmin(np.abs(lams_curve - lam)))

        frames.append(go.Frame(
            name=f"{lam:.2f}",
            traces=dyn_ix,
            data=[
                go.Scatter(x=cx, y=cy),
                go.Scatter(x=[w[0]], y=[w[1]]),
                go.Scatter(x=[lam, lam], y=[0, ymax]),
                go.Scatter(x=[lams_curve[idx]], y=[ED_curve[idx]],
                        text=[f"ED={ED_curve[idx]:.3f}"]),
                go.Scatter(x=[lams_curve[idx]], y=[lams_curve[idx]*EW_curve[idx]],
                        text=[f"λEw={(lams_curve[idx]*EW_curve[idx]):.3f}"]),
                go.Scatter(x=[lams_curve[idx]], y=[E_curve[idx]],
                        text=[f"E={E_curve[idx]:.3f}"]),
            ]
        ))
    fig.frames = frames

    # Menyiapkan tampilan awal pada λ pertama
    lam0 = float(lams[0])
    w_init, ED0, EW0, E0 = func(lam0, A, w0)

    i0 = int(np.argmin(np.abs(lams_curve - lam0)))
    if func is ridge_solution:
        c0 = np.linalg.norm(w_init)
        fig.data[dyn_ix[0]].update(x=c0*np.cos(t), y=c0*np.sin(t))
    elif func is lasso_solution:
        c0 = np.linalg.norm(w_init, ord=1)
        cx_init = c0 * np.array([1, 0, -1, 0, 1])
        cy_init = c0 * np.array([0, 1, 0, -1, 0])
        fig.data[dyn_ix[0]].update(x=cx_init, y=cy_init)
    fig.data[dyn_ix[1]].update(x=[w_init[0]], y=[w_init[1]])
    fig.data[dyn_ix[2]].update(x=[lam0, lam0], y=[0, ymax])
    fig.data[dyn_ix[3]].update(x=[lams_curve[i0]], y=[ED_curve[i0]], text=[f"ED={ED_curve[i0]:.3f}"])
    fig.data[dyn_ix[4]].update(x=[lams_curve[i0]], y=[lams_curve[i0]*EW_curve[i0]], text=[f"λEw={(lams_curve[i0]*EW_curve[i0]):.3f}"])
    fig.data[dyn_ix[5]].update(x=[lams_curve[i0]], y=[E_curve[i0]], text=[f"E={E_curve[i0]:.3f}"])  # sumbu-x dikunci pada λ

    slider_steps = [{"label": f"λ = {lam:.2f}", "method": "animate",
                    "args": [[f"{lam:.2f}"],
                            {"mode": "immediate",
                            "frame": {"duration": 0, "redraw": True},
                            "transition": {"duration": 0}}]} for lam in lams]

    updatemenus = [dict(
        type="buttons", showactive=True, x=0.05, y=0, xanchor="left", yanchor="bottom",
        direction="left",
        buttons=[
            dict(label="▶", method="animate",
                args=[None, {"fromcurrent": True,
                            "frame": {"duration": 600, "redraw": True},
                            "transition": {"duration": 50}}]),
            dict(label="⏸", method="animate",
                args=[[None], {"mode": "immediate",
                                "frame": {"duration": 0, "redraw": False},
                                "transition": {"duration": 0}}]),
        ]
    )]

    fig.update_layout(
        sliders=[dict(
            active=0,
            steps=slider_steps,
            pad=dict(l=120, t=60)
        )],
        updatemenus=updatemenus,
        uirevision="keep"
    )

    fig.update_xaxes(title_text="w₁", row=1, col=1, zeroline=False, range=[-10, 10], title=dict(standoff=20))
    fig.update_yaxes(title_text="w₂", row=1, col=1, zeroline=False, range=[-10, 10], title=dict(standoff=20), scaleanchor="x", scaleratio=1)
    fig.update_xaxes(title_text="λ (kekuatan regularisasi)", row=1, col=2, title=dict(standoff=20))
    fig.update_xaxes(row=1, col=1, range=[w0[0]-10, w0[0]+10])
    fig.update_yaxes(row=1, col=1, range=[w0[1]-10, w0[1]+10])

    zmin, zmax = 0.0, float(np.percentile(Z_ED, 95))

    return fig

In [ ]:
plot_reg_with_regularization(ridge_solution).show()

**Cara membaca grafik interaktif di atas.**

Panel kiri menampilkan ruang parameter: kontur galat, lingkaran batas L2, dan titik solusi. Panel kanan menampilkan tiga kurva terhadap $\lambda$:

| Kurva | Arti |
|---|---|
| $E_D(\hat{w}(\lambda))$ | Galat pada data — **naik** seiring $\lambda$ membesar |
| $\lambda \cdot E_W(\hat{w}(\lambda))$ | Sumbangan penalti terhadap fungsi tujuan |
| $E(\hat{w}(\lambda))$ | Jumlah keduanya, yaitu fungsi tujuan total |

> **Inilah pertukarannya.** Menaikkan $\lambda$ selalu memperburuk kecocokan pada data latih, tetapi menghasilkan bobot yang lebih kecil dan model yang lebih sederhana. Yang kita cari adalah keseimbangan di antara keduanya.

### 3.2 Regularisasi L1: Lasso Regression

Grafik berikut sama persis susunannya, hanya daerah batasnya yang berbeda: kini berbentuk **belah ketupat** ($\|\mathbf{w}\|_1 = c(\lambda)$), bukan lingkaran.

> **Perhatikan baik-baik.** Amati di mana titik solusi menyentuh belah ketupat itu ketika $\lambda$ diperbesar. Bila titiknya berhenti tepat di salah satu **sudut**, artinya salah satu koefisien menjadi tepat nol — inilah mekanisme pemilihan fitur pada Lasso.


In [ ]:
w0 = np.array([3.2, 0.9])
theta = np.deg2rad(25)
R = np.array([[np.cos(theta), -np.sin(theta)],
                [np.sin(theta),  np.cos(theta)]])
A = R @ np.diag([6.0, 1.0]) @ R.T * 2.5

fig = plot_E_D()

lam = 0.0

w, ED, EW, J = lasso_solution(lam, A, w0)
c = np.linalg.norm(w, ord=1)
cx = c * np.array([1, 0, -1, 0, 1])
cy = c * np.array([0, 1, 0, -1, 0])

fig.add_trace(go.Scatter(x=cx, y=cy, mode="lines",
                       line=dict(width=5, color="darkmagenta"), name="‖w‖₂ = c(λ)"))
fig.add_trace(go.Scatter(x=[w[0]], y=[w[1]], mode="markers",
                       marker=dict(size=12, symbol="x", color="teal"), name="ŵ(λ)"))

fig.show()

In [ ]:

fig = plot_reg_with_regularization(lasso_solution).show()

**Bandingkan dengan visualisasi Ridge sebelumnya.** Perbedaan satu-satunya ada pada bentuk daerah batasnya, tetapi akibatnya sangat berbeda: pada Lasso, solusi sering berhenti tepat di sudut belah ketupat, dan sudut berarti salah satu koefisien bernilai **tepat nol**.

---
## 4. Dampak Regularisasi

Bagian ini menunjukkan pengaruh regularisasi pada kasus nyata: mencocokkan polinomial berderajat tinggi pada data yang sebenarnya berpola $2\sin(2\pi x)$, dengan data latih yang sedikit dan berderau.

Tanpa regularisasi, polinomial berderajat 8 akan meliuk-liuk mengikuti derau — inilah overfitting. Dengan regularisasi yang tepat, kurvanya kembali mulus dan mendekati pola sebenarnya.

Grafik interaktif berikut memiliki dua panel:

| Panel | Isinya |
|---|---|
| Kiri | Data latih, data uji, kurva sebenarnya, dan kurva hasil pelatihan |
| Kanan | MSE data latih dan MSE data uji terhadap $\lambda$ |

**Geser penggeser $\lambda$** dan amati keduanya bersamaan. Perhatikan tiga hal:

1. Pada $\lambda$ sangat kecil, kurva kiri meliuk berlebihan; MSE latih rendah tetapi MSE uji tinggi → **overfitting**.
2. Pada $\lambda$ sangat besar, kurva kiri menjadi hampir mendatar; kedua MSE tinggi → **underfitting**.
3. Di antara keduanya ada $\lambda$ yang membuat MSE uji paling rendah → **titik seimbang** yang kita cari.

> **Ini pola yang sama** dengan kurva latih–validasi pada Pertemuan 03, hanya saja di sini yang kita setel adalah kekuatan regularisasi, bukan kompleksitas model secara langsung.


In [ ]:
from sklearn.linear_model import Lasso, Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

def plot_regularized_model_performance(regularized_model, ratio=0.8, degree=6):
    X = np.linspace(0, 1, 40).reshape(-1, 1)
    y_true = 2* np.sin(2*np.pi*X).ravel()
    y = y_true + np.random.normal(0, 0.7, X.shape[0])


    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=ratio, random_state=189
    )

    X_plot = np.linspace(0, 1, 400).reshape(-1, 1)
    y_true_plot = 2* np.sin(2*np.pi*X_plot).ravel()

    poly = PolynomialFeatures(degree=degree, include_bias=False)
    X_train_poly = poly.fit_transform(X_train)
    X_test_poly  = poly.transform(X_test)
    X_plot_poly  = poly.transform(X_plot)

    lambdas = np.array([100, 35, 20, 1, 0.2, 0.05, 0.01, 0.003, 0.0009, 0.00025, 0.00013])      # daftar nilai λ yang dicoba
    lambda_slider = lambdas.copy()

    train_mse, test_mse, pen_curve = [], [], []
    fits_over_lambda = []
    models = []
    for a in lambdas:
        model_instance = regularized_model(alpha=a, fit_intercept=True)
        model_instance.fit(X_train_poly, y_train)
        models.append(model_instance)

        # Menghitung galat pada data latih dan data uji
        y_tr = model_instance.predict(X_train_poly)
        y_te = model_instance.predict(X_test_poly)
        train_mse.append(mean_squared_error(y_train, y_tr))
        test_mse.append(mean_squared_error(y_test,  y_te))

        # Penalti bobot (intersep tidak ikut dihukum)
        w = model_instance.coef_.ravel()
        pen_curve.append(float(a * np.sum(w**2)))

        # Kurva hasil pelatihan pada kisi rapat (panel kiri)
        fits_over_lambda.append(model_instance.predict(X_plot_poly))

    train_mse = np.array(train_mse)
    test_mse  = np.array(test_mse)
    pen_curve = np.array(pen_curve)
    train_obj = train_mse + pen_curve
    test_obj  = test_mse  + pen_curve
    right_ymax = 1.05 * float(max(train_obj.max(), test_obj.max()))
    right_ymin = 0.95 * float(min(train_obj.min(), test_obj.min()))

    fits_over_alpha = np.vstack(fits_over_lambda)

    y_left_all = np.concatenate([y_train, y_test, y_true_plot])
    ymin_left, ymax_left = float(y_left_all.min()), float(y_left_all.max())
    pad = max(0.25, 0.10*(ymax_left - ymin_left))
    ymin_left -= pad; ymax_left += pad

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("Kurva Hasil Pelatihan beserta Data Latih dan Uji",
                        "MSE Latih dan Uji terhadap λ"),
        specs=[[{"type":"xy"}, {"type":"xy"}]],
        column_widths=[0.55, 0.45]
    )

    # PANEL KIRI (unsur tetap)
    fig.add_trace(go.Scatter(x=X_train.ravel(), y=y_train, mode='markers',
                            name='Data latih', marker=dict(size=7)),
                row=1, col=1)
    fig.add_trace(go.Scatter(x=X_test.ravel(), y=y_test, mode='markers',
                            name='Data uji', marker=dict(size=7, symbol='square')),
                row=1, col=1)
    fig.add_trace(go.Scatter(x=X_plot.ravel(), y=y_true_plot, mode='lines',
                            name='Fungsi sebenarnya', line=dict(width=2, dash='dot')),
                row=1, col=1)

    # PANEL KIRI (unsur yang berubah)
    fit_tr = go.Scatter(x=[], y=[], mode='lines',
                        name='Kurva hasil pelatihan', line=dict(color='red', width=3))
    fig.add_trace(fit_tr, row=1, col=1)

    # PANEL KANAN (unsur tetap)
    fig.add_trace(go.Scatter(x=lambdas, y=train_mse, mode='lines',
                            name='MSE data latih', line=dict(width=3)),
                row=1, col=2)
    fig.add_trace(go.Scatter(x=lambdas, y=test_mse, mode='lines',
                            name='MSE data uji', line=dict(width=3)),
                row=1, col=2)

    # PANEL KANAN (unsur yang berubah): garis λ dan penandanya
    vline_tr = go.Scatter(x=[], y=[], mode='lines',
                        line=dict(color='crimson', dash='dot', width=2),
                        showlegend=False)
    mk_tr_mse = go.Scatter(x=[], y=[], mode='markers+text', text=[""],
                        marker=dict(size=9), textposition="top center", showlegend=False)
    mk_te_mse = go.Scatter(x=[], y=[], mode='markers+text', text=[""],
                        marker=dict(size=9), textposition="top center", showlegend=False)

    fig.add_trace(vline_tr, row=1, col=2)
    fig.add_trace(mk_tr_mse, row=1, col=2)
    fig.add_trace(mk_te_mse, row=1, col=2)

    dyn_ix = [3, 6, 7, 8]


    def nearest_idx(arr, val): return int(np.argmin(np.abs(arr - val)))

    # ---------- Bingkai animasi: tiap λ memperbarui kurva kiri dan penanda kanan ----------
    frames = []
    for i, a in enumerate(lambda_slider):
        j = nearest_idx(lambdas, a)
        frames.append(go.Frame(
            name=f"{a:.5f}",
            traces=dyn_ix,
            data=[
                go.Scatter(x=X_plot.ravel(), y=fits_over_alpha[j]),
                go.Scatter(x=[a, a], y=[0, right_ymax]),
                go.Scatter(x=[lambdas[j]], y=[train_mse[j]]),
                go.Scatter(x=[lambdas[j]], y=[test_mse[j]])
            ],
        ))
    fig.frames = frames

    alpha0 = float(lambda_slider[0])
    j0 = nearest_idx(lambdas, alpha0)
    fig.data[dyn_ix[0]].update(x=X_plot.ravel(), y=fits_over_alpha[j0])
    fig.data[dyn_ix[1]].update(x=[alpha0, alpha0], y=[0, right_ymax])
    fig.data[dyn_ix[2]].update(x=[lambdas[j0]], y=[train_mse[j0]])
    fig.data[dyn_ix[3]].update(x=[lambdas[j0]], y=[test_mse[j0]])

    slider_steps = [{"label": f"λ = {a:.2f}", "method": "animate",
                    "args": [[f"{a:.5f}"],
                            {"mode": "immediate",
                            "frame": {"duration": 0, "redraw": True},
                            "transition": {"duration": 0}}]} for a in lambda_slider]

    updatemenus = [dict(
        type="buttons", showactive=True, x=0.05, y=0, xanchor="left", yanchor="bottom",
        direction="left",
        buttons=[
            dict(label="▶", method="animate",
                args=[None, {"fromcurrent": True,
                            "frame": {"duration": 600, "redraw": True},
                            "transition": {"duration": 50}}]),
            dict(label="⏸", method="animate",
                args=[[None], {"mode": "immediate",
                                "frame": {"duration": 0, "redraw": False},
                                "transition": {"duration": 0}}]),
        ]
    )]

    fig.update_layout(
        sliders=[dict(
            active=0,
            steps=slider_steps,
            pad=dict(l=120, t=60)
        )],
        updatemenus=updatemenus
    )

    fig.update_xaxes(title_text='x', row=1, col=1, range=[0, 1])
    fig.update_yaxes(title_text='y', row=1, col=1, range=[ymin_left-0.5, ymax_left+0.5])
    fig.update_xaxes(title_text='λ (skala logaritma)', type='log', row=1, col=2)
    fig.update_yaxes(title_text='Galat (MSE)', row=1, col=2, range=[-0.5, right_ymax+0.5], rangemode='tozero')
    fig.update_layout(template="plotly_white")
    return fig, models

In [ ]:
# Menyusun tabel koefisien untuk setiap nilai lambda yang dicoba
def display_coefficients(models):
    models_df = pd.DataFrame(models, columns=["Model"])

    coefficients_data = {}
    for index, row in models_df.iterrows():
        model = row["Model"]
        alpha_value = model.alpha
        coefficients = model.coef_.ravel()

        coefficients_data[f'lambda={alpha_value:.5e}'] = coefficients

    coefficients_df = pd.DataFrame(coefficients_data)
    coefficients_df = coefficients_df.rename_axis("Derajat Polinomial")
    return coefficients_df

In [ ]:
fig, models = plot_regularized_model_performance(Ridge, ratio=0.85, degree=8)

fig.show()

**Geser penggeser $\lambda$ dari kanan ke kiri** (dari kecil ke besar) dan amati kedua panel bersamaan.

Perhatikan bahwa MSE data latih **selalu naik** seiring $\lambda$ membesar, sedangkan MSE data uji **turun dulu lalu naik**. Titik terendah kurva MSE uji itulah $\lambda$ terbaik.

In [ ]:
display_coefficients(models)

Tabel di atas menampilkan nilai koefisien untuk setiap $\lambda$. Barisnya adalah derajat polinomial, kolomnya adalah nilai $\lambda$ dari besar ke kecil.

**Perhatikan besarnya angka.** Pada $\lambda$ yang sangat kecil, koefisien bisa mencapai puluhan atau ratusan — inilah ciri khas model yang overfitting: bobot besar yang saling meniadakan untuk memaksa kurva melewati setiap titik data.

In [ ]:
coefficients_df = display_coefficients(models)
coefficients_df.T.plot(kind='line', figsize=(10, 6), marker='o')
plt.title("Besar Koefisien terhadap Lambda")
plt.xlabel("Lambda (dari besar ke kecil)")
plt.ylabel("Besar koefisien")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend(title="Koefisien", bbox_to_anchor=(1.05, 1), loc='upper left')

plt.xticks(ticks=range(len(coefficients_df.columns)), labels=coefficients_df.columns, rotation=45)

plt.tight_layout()
plt.show()


Grafik ini merangkum penyusutan koefisien pada **Ridge**. Perhatikan bahwa seluruh garis bergerak menuju nol seiring $\lambda$ membesar, tetapi **tidak ada yang benar-benar mencapai nol**. Inilah ciri khas regularisasi L2.

In [ ]:
fig, models = plot_regularized_model_performance(Lasso, ratio=0.85, degree=8)

fig.show()

In [ ]:
display_coefficients(models)

Bandingkan tabel ini dengan tabel Ridge sebelumnya. Cari nilai yang **tepat nol** (atau sangat mendekati nol) — pada Lasso jumlahnya jauh lebih banyak.

In [ ]:
coefficients_df = display_coefficients(models)
coefficients_df.T.plot(kind='line', figsize=(10, 6), marker='o')
plt.title("Besar Koefisien terhadap Lambda")
plt.xlabel("Lambda (dari besar ke kecil)")
plt.ylabel("Besar koefisien")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend(title="Koefisien", bbox_to_anchor=(1.05, 1), loc='upper left')

plt.xticks(ticks=range(len(coefficients_df.columns)), labels=coefficients_df.columns, rotation=45)

plt.tight_layout()
plt.show()

Grafik penyusutan koefisien pada **Lasso**. Bandingkan dengan grafik Ridge: di sini banyak garis yang **rata di angka nol** untuk $\lambda$ besar, artinya fitur-fitur itu benar-benar dibuang dari model.

> **Inilah pemilihan fitur otomatis.** Lasso tidak sekadar mengecilkan pengaruh fitur yang kurang berguna, melainkan menghapusnya sama sekali. Sifat ini sangat berharga ketika kita punya ratusan fitur dan ingin tahu mana yang benar-benar penting.

---
## 5. Ill-Conditioning (Matriks yang Buruk Kondisinya)

Selain mencegah overfitting, regularisasi juga menyelesaikan masalah **numerik**.

**Apa itu ill-conditioning?** Solusi regresi linear diperoleh dari persamaan normal $\mathbf{w} = (\mathbb{X}^T\mathbb{X})^{-1}\mathbb{X}^T\mathbf{t}$. Bila dua fitur hampir **kolinear** — yaitu nilainya nyaris identik atau salah satunya hampir merupakan kelipatan yang lain — maka matriks $\mathbb{X}^T\mathbb{X}$ menjadi hampir singular dan sangat sulit dibalik secara stabil.

**Bilangan kondisi** ($\kappa$) mengukur seberapa parah masalah ini:

| Nilai $\kappa$ | Tafsirannya |
|---|---|
| Dekat 1 | Matriks berkondisi baik, solusinya stabil |
| $10^3$ – $10^6$ | Mulai mengkhawatirkan |
| $> 10^8$ | Ill-conditioned; solusinya sangat peka terhadap derau sekecil apa pun |

**Akibatnya di praktik.** Perubahan sangat kecil pada data dapat menghasilkan koefisien yang sangat berbeda, bahkan berlawanan tanda dan berukuran raksasa. Prediksinya boleh jadi masih baik, tetapi koefisiennya tidak dapat ditafsirkan sama sekali.

**Bagaimana Ridge menolong.** Ridge menyelesaikan $(\mathbb{X}^T\mathbb{X} + \lambda\mathbf{I})^{-1}\mathbb{X}^T\mathbf{t}$. Penambahan $\lambda$ pada diagonal menjauhkan matriks itu dari keadaan singular, sehingga solusinya kembali stabil. Inilah alasan Ridge kadang disebut *ridge* — ia menambahkan sebuah "punggungan" pada diagonal matriks.

Sel berikut membuat contohnya: dua fitur yang selisihnya hanya $10^{-4}$, sehingga praktis identik. Bandingkan bobot hasil persamaan normal dengan bobot hasil Ridge, dan bandingkan keduanya terhadap bobot sebenarnya yaitu $[3, 2, -1]$.


In [ ]:
# --- 1. Membuat matriks rancangan yang fiturnya hampir kolinear ------------
N, D = 500, 2                       # satu bias + 2 fitur
X = np.ones((N, D + 1))             # kolom pertama = bias
x1 = np.random.randn(N)
x2 = x1 + 1e-4 * np.random.randn(N) # fitur yang hampir identik dengan x1
X[:, 1] = x1
X[:, 2] = x2

t = 3 + 2*x1 - 1*x2 + 0.1*np.random.randn(N)  # bobot sebenarnya: [3, 2, -1]

# --- 2. Solusi lewat persamaan normal (tanpa regularisasi) ----------------
XtX   = X.T @ X
Xty   = X.T @ t
w_ne  = np.linalg.solve(XtX, Xty)

# --- 3. Menghitung bilangan kondisi ---------------------------------------
u, s, vt = np.linalg.svd(X, full_matrices=False)
kappa = s.max() / s.min()

# --- 4. Solusi Ridge dengan λ = 1e-2 --------------------------------------
lam   = 1e-2
w_ridge = np.linalg.solve(XtX + lam*np.eye(D+1), Xty)

print("Bilangan kondisi cond(XᵀX) =", kappa**2)   # dikuadratkan karena XᵀX mengkuadratkan nilai singular
print("Bobot lewat persamaan normal:", w_ne)
print("Bobot lewat Ridge           :", w_ridge)

**Bacalah ketiga keluaran di atas.**

Bilangan kondisi bernilai sangat besar — inilah tanda **ill-conditioning**. Bandingkan kedua vektor bobot terhadap nilai sebenarnya $[3, 2, -1]$:

* **Persamaan normal** menghasilkan koefisien yang jauh menyimpang dan berukuran besar, karena kedua fitur hampir identik sehingga model bebas memberi bobot besar pada satu fitur dan bobot besar berlawanan tanda pada fitur lainnya.
* **Ridge** menghasilkan koefisien yang jauh lebih wajar dan stabil, karena penambahan $\lambda\mathbf{I}$ pada diagonal menghilangkan kebebasan yang merusak itu.

> **Catatan penting.** Pada kasus seperti ini, **prediksi** dari kedua model bisa sama-sama baik. Yang rusak adalah **penafsiran koefisiennya**. Bila tujuan Anda hanya memprediksi, ill-conditioning mungkin tidak terasa; tetapi bila Anda ingin menjelaskan pengaruh tiap fitur, hasilnya sama sekali tidak dapat dipercaya.

---
## 6. Penutup

### Ringkasan

**Geometri**

* Prediksi $\mathbf{Y} = \mathbb{X}\mathbf{w}$ selalu terletak di dalam $\text{span}(\mathbb{X})$.
* Solusi terbaik adalah **proyeksi ortogonal** dari $\mathbf{t}$ ke subruang itu, sehingga residual $\mathbf{e}$ tegak lurus terhadap seluruh kolom $\mathbb{X}$.
* Syarat $\mathbb{X}^T\mathbf{e} = 0$ inilah yang bila diuraikan menghasilkan persamaan normal.

**Residual**

* Residual yang tersebar acak di sekitar nol menandakan model sudah cocok.
* Pola menyerupai kipas menandakan **heteroskedastisitas**, yaitu ragam galat yang tidak tetap.

**Regularisasi**

* Fungsi tujuan menjadi $E(\mathbf{w}) = E_D(\mathbf{w}) + \lambda E_W(\mathbf{w})$, dengan $\lambda$ mengatur kekuatan penaltinya.
* **Ridge (L2)** memakai $\sum w_j^2$; seluruh koefisien menyusut merata tetapi jarang mencapai nol.
* **Lasso (L1)** memakai $\sum |w_j|$; sebagian koefisien menjadi tepat nol, sehingga sekaligus melakukan pemilihan fitur.
* Perbedaan itu berasal dari bentuk daerah batasnya: lingkaran yang mulus versus belah ketupat yang bersudut.
* $\lambda$ terlalu kecil menyebabkan overfitting, terlalu besar menyebabkan underfitting; nilai terbaiknya dipilih dengan data validasi.

**Ill-conditioning**

* Fitur yang hampir kolinear membuat $\mathbb{X}^T\mathbb{X}$ nyaris singular, dan koefisiennya menjadi raksasa serta tidak stabil.
* Ridge menambahkan $\lambda\mathbf{I}$ pada diagonal sehingga masalah numerik itu teratasi.

### Latihan Mandiri

1. Pada grafik geometri tiga dimensi, putar tampilannya sampai bidang terlihat dari samping. Buktikan secara numerik bahwa residual tegak lurus terhadap kedua vektor kolom dengan memeriksa bahwa hasil kali titiknya mendekati nol.
2. Pada visualisasi Ridge, geser $\lambda$ dari 0 sampai 15. Bagaimana perubahan jari-jari lingkaran, dan ke arah mana solusinya bergerak?
3. Ulangi latihan nomor 2 pada visualisasi Lasso. Pada nilai $\lambda$ berapa solusinya mulai menyentuh sudut belah ketupat? Apa arti kejadian itu bagi koefisiennya?
4. Pada Bagian 4, jalankan `plot_regularized_model_performance` dengan `degree=3` lalu `degree=12`. Bagaimana pengaruh derajat polinomial terhadap letak $\lambda$ terbaik?
5. Amati tabel koefisien hasil Ridge dan hasil Lasso. Hitung berapa banyak koefisien yang bernilai **tepat nol** pada masing-masing, untuk $\lambda$ terbesar. Apakah hasilnya sesuai dengan teori?
6. Pada Bagian 5, ubah `1e-4` menjadi `1e-2` lalu `1e-8`, dan amati perubahan bilangan kondisinya. Pada nilai berapa bobot hasil persamaan normal mulai kacau?
7. Masih pada Bagian 5, coba beberapa nilai `lam` untuk Ridge: `1e-6`, `1e-2`, dan `1`. Adakah pertukaran antara kestabilan koefisien dan kedekatannya dengan bobot sebenarnya?
8. Jelaskan dengan kalimat Anda sendiri: mengapa dalam kasus ill-conditioning prediksi model masih bisa akurat, padahal koefisiennya sama sekali tidak dapat dipercaya?

### Bacaan Lanjutan

* Ridge dan Lasso di scikit-learn: https://scikit-learn.org/stable/modules/linear_model.html
* Pemilihan $\lambda$ dengan validasi silang: `RidgeCV` dan `LassoCV`
